In [ ]:
!pip install -q tensorflow pandas numpy scikit-learn matplotlib seaborn

In [ ]:
from google.colab import files
print("Upload these files:")
print("1. nasa_train.csv")
print("2. nasa_val.csv")
print("3. nasa_test.csv")
print("4. nasa_scaler.pkl")
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
train_df = pd.read_csv('nasa_train.csv')
val_df = pd.read_csv('nasa_val.csv')
test_df = pd.read_csv('nasa_test.csv')

# Load scaler
with open('nasa_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print("✓ Train:", train_df.shape)
print("✓ Val:", val_df.shape)
print("✓ Test:", test_df.shape)

# Prepare data
X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values
X_val = val_df.iloc[:, :-1].values
y_val = val_df.iloc[:, -1].values
X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values

# Reshape for models
X_train_reshaped = X_train.reshape(-1, 10, 6)
X_val_reshaped = X_val.reshape(-1, 10, 6)
X_test_reshaped = X_test.reshape(-1, 10, 6)

print("✓ Data prepared and reshaped")

In [ ]:
def build_cnn():
    model = keras.Sequential([
        layers.Input(shape=(10, 6)),
        layers.Conv1D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        layers.Conv1D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

cnn_model = build_cnn()
cnn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
print("✓ CNN Model built")

In [ ]:
print("Training CNN... (will take 5-10 minutes)")
history_cnn = cnn_model.fit(
    X_train_reshaped, y_train,
    validation_data=(X_val_reshaped, y_val),
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
    ]
)
print("✓ CNN training complete")

In [ ]:
y_pred_cnn = cnn_model.predict(X_test_reshaped).flatten()
mae_cnn = mean_absolute_error(y_test, y_pred_cnn)
rmse_cnn = np.sqrt(mean_squared_error(y_test, y_pred_cnn))
mape_cnn = np.mean(np.abs((y_test - y_pred_cnn) / y_test)) * 100
r2_cnn = r2_score(y_test, y_pred_cnn)

print("="*50)
print("CNN MODEL RESULTS")
print("="*50)
print(f"MAE:  {mae_cnn:.6f}")
print(f"RMSE: {rmse_cnn:.6f}")
print(f"MAPE: {mape_cnn:.4f}%")
print(f"R²:   {r2_cnn:.6f}")
print("="*50)

In [ ]:
def build_lstm():
    model = keras.Sequential([
        layers.Input(shape=(10, 6)),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

lstm_model = build_lstm()
lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
print("✓ LSTM Model built")


In [ ]:
print("Training LSTM... (will take 10-15 minutes)")
history_lstm = lstm_model.fit(
    X_train_reshaped, y_train,
    validation_data=(X_val_reshaped, y_val),
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
    ]
)
print("✓ LSTM training complete")

In [ ]:
y_pred_lstm = lstm_model.predict(X_test_reshaped).flatten()
mae_lstm = mean_absolute_error(y_test, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))
mape_lstm = np.mean(np.abs((y_test - y_pred_lstm) / y_test)) * 100
r2_lstm = r2_score(y_test, y_pred_lstm)

print("="*50)
print("LSTM MODEL RESULTS")
print("="*50)
print(f"MAE:  {mae_lstm:.6f}")
print(f"RMSE: {rmse_lstm:.6f}")
print(f"MAPE: {mape_lstm:.4f}%")
print(f"R²:   {r2_lstm:.6f}")
print("="*50)

In [ ]:
def build_hybrid():
    model = keras.Sequential([
        layers.Input(shape=(10, 6)),
        layers.Conv1D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(32, return_sequences=False),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

hybrid_model = build_hybrid()
hybrid_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
print("✓ Hybrid Model built")

In [ ]:
print("Training Hybrid... (will take 10-15 minutes)")
history_hybrid = hybrid_model.fit(
    X_train_reshaped, y_train,
    validation_data=(X_val_reshaped, y_val),
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
    ]
)
print("✓ Hybrid training complete")

In [ ]:
y_pred_hybrid = hybrid_model.predict(X_test_reshaped).flatten()
mae_hybrid = mean_absolute_error(y_test, y_pred_hybrid)
rmse_hybrid = np.sqrt(mean_squared_error(y_test, y_pred_hybrid))
mape_hybrid = np.mean(np.abs((y_test - y_pred_hybrid) / y_test)) * 100
r2_hybrid = r2_score(y_test, y_pred_hybrid)

print("="*50)
print("HYBRID MODEL RESULTS")
print("="*50)
print(f"MAE:  {mae_hybrid:.6f}")
print(f"RMSE: {rmse_hybrid:.6f}")
print(f"MAPE: {mape_hybrid:.4f}%")
print(f"R²:   {r2_hybrid:.6f}")
print("="*50)

In [ ]:
comparison = pd.DataFrame({
    'Model': ['CNN', 'LSTM', 'Hybrid'],
    'MAE': [mae_cnn, mae_lstm, mae_hybrid],
    'RMSE': [rmse_cnn, rmse_lstm, rmse_hybrid],
    'MAPE (%)': [mape_cnn, mape_lstm, mape_hybrid],
    'R²': [r2_cnn, r2_lstm, r2_hybrid]
})

print("\n" + "="*70)
print("FINAL COMPARISON")
print("="*70)
print(comparison.to_string(index=False))
print("="*70)

# Find best model
best_model_name = comparison.loc[comparison['MAPE (%)'].idxmin(), 'Model']
print(f"\n🏆 BEST MODEL: {best_model_name}")

In [ ]:
cnn_model.save('cnn_model.h5')
lstm_model.save('lstm_model.h5')
hybrid_model.save('hybrid_model.h5')

with open('cnn_model.pkl', 'wb') as f:
    pickle.dump(cnn_model, f)
with open('lstm_model.pkl', 'wb') as f:
    pickle.dump(lstm_model, f)
with open('hybrid_model.pkl', 'wb') as f:
    pickle.dump(hybrid_model, f)

print("✓ All models saved")

In [ ]:
from google.colab import files

# Download best model files
files.download('cnn_model.h5')
files.download('lstm_model.h5')
files.download('hybrid_model.h5')
files.download('cnn_model.pkl')
files.download('lstm_model.pkl')
files.download('hybrid_model.pkl')

print("✓ Download complete - NASA PART DONE!")